<a href="https://colab.research.google.com/github/Nithin025au/IDSS-Lab/blob/main/1_Rule_based_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import pandas as pd
df=pd.read_excel("gezinomi.xlsx")
df

,SaleId,SaleDate,CheckInDate,Price,ConceptName,SaleCityName,CInDay,SaleCheckInDayDiff,Seasons
0,415122,2022-12-03,2022-12-03,79.304029,Herşey Dahil,Antalya,Saturday,0,Low
1,415103,2022-12-03,2022-12-03,45.970696,Yarım Pansiyon,Antalya,Saturday,0,Low
2,404034,2022-09-12,2022-09-13,77.838828,Herşey Dahil,Antalya,Tuesday,1,High
3,415094,2022-12-03,2022-12-10,222.710623,Yarım Pansiyon,İzmir,Saturday,7,Low
4,414951,2022-12-01,2022-12-03,140.476190,Yarım Pansiyon,İzmir,Saturday,2,Low
...,...,...,...,...,...,...,...,...,...
59159,51817,2016-01-05,2016-10-10,54.304636,Herşey Dahil,Antalya,Monday,279,Low
59160,51816,2016-01-05,2016-10-10,54.304636,Herşey Dahil,Antalya,Monday,279,Low
59161,51814,2016-01-05,2016-01-06,40.562914,Herşey Dahil,Diğer,Wednesday,1,Low
59162,51736,2016-01-04,2016-01-05,69.847682,Yarım Pansiyon,Diğer,Tuesday,1,Low


In [47]:
def assign_category(price):
    if price < 50:
        return 'Low Price'
    elif 50 <= price < 150:
        return 'Medium Price'
    else:
        return 'High Price'

df['Category'] = df['Price'].apply(assign_category)
df.head()

,SaleId,SaleDate,CheckInDate,Price,ConceptName,SaleCityName,CInDay,SaleCheckInDayDiff,Seasons,Category
0,415122,2022-12-03,2022-12-03,79.304029,Herşey Dahil,Antalya,Saturday,0,Low,Medium Price
1,415103,2022-12-03,2022-12-03,45.970696,Yarım Pansiyon,Antalya,Saturday,0,Low,Low Price
2,404034,2022-09-12,2022-09-13,77.838828,Herşey Dahil,Antalya,Tuesday,1,High,Medium Price
3,415094,2022-12-03,2022-12-10,222.710623,Yarım Pansiyon,İzmir,Saturday,7,Low,High Price
4,414951,2022-12-01,2022-12-03,140.476190,Yarım Pansiyon,İzmir,Saturday,2,Low,Medium Price


In [48]:
def assign_concept_category(concept_name):
    if 'Herşey Dahil' in concept_name:
        return 'All Inclusive'
    elif 'Yarım Pansiyon' in concept_name:
        return 'Half Board'
    else:
        return 'Other Concept'

df['Concept_Category'] = df['ConceptName'].apply(assign_concept_category)
display(df.head())

,SaleId,SaleDate,CheckInDate,Price,ConceptName,SaleCityName,CInDay,SaleCheckInDayDiff,Seasons,Category,Concept_Category
0,415122,2022-12-03,2022-12-03,79.304029,Herşey Dahil,Antalya,Saturday,0,Low,Medium Price,All Inclusive
1,415103,2022-12-03,2022-12-03,45.970696,Yarım Pansiyon,Antalya,Saturday,0,Low,Low Price,Half Board
2,404034,2022-09-12,2022-09-13,77.838828,Herşey Dahil,Antalya,Tuesday,1,High,Medium Price,All Inclusive
3,415094,2022-12-03,2022-12-10,222.710623,Yarım Pansiyon,İzmir,Saturday,7,Low,High Price,Half Board
4,414951,2022-12-01,2022-12-03,140.476190,Yarım Pansiyon,İzmir,Saturday,2,Low,Medium Price,Half Board


In [49]:
def categorize_stay_duration(diff):
    if diff < 3:
        return 'Short Stay'
    elif 3 <= diff <= 7:
        return 'Medium Stay'
    else:
        return 'Long Stay'

df['Stay_Duration_Category'] = df['SaleCheckInDayDiff'].apply(categorize_stay_duration)
display(df.head())

,SaleId,SaleDate,CheckInDate,Price,ConceptName,SaleCityName,CInDay,SaleCheckInDayDiff,Seasons,Category,Concept_Category,Stay_Duration_Category
0,415122,2022-12-03,2022-12-03,79.304029,Herşey Dahil,Antalya,Saturday,0,Low,Medium Price,All Inclusive,Short Stay
1,415103,2022-12-03,2022-12-03,45.970696,Yarım Pansiyon,Antalya,Saturday,0,Low,Low Price,Half Board,Short Stay
2,404034,2022-09-12,2022-09-13,77.838828,Herşey Dahil,Antalya,Tuesday,1,High,Medium Price,All Inclusive,Short Stay
3,415094,2022-12-03,2022-12-10,222.710623,Yarım Pansiyon,İzmir,Saturday,7,Low,High Price,Half Board,Medium Stay
4,414951,2022-12-01,2022-12-03,140.476190,Yarım Pansiyon,İzmir,Saturday,2,Low,Medium Price,Half Board,Short Stay


In [50]:
import numpy as np

# Create 'Discounted_Price' column (re-adding missing dependency)
df['Discounted_Price'] = df.apply(
    lambda row: row['Price'] * 0.9 if row['Seasons'] == 'Low' else row['Price'], axis=1
)

# Define function to calculate numerical impact (re-adding missing dependency)
def calculate_numerical_impact(row):
    price = row['Discounted_Price']
    stay_duration = row['Stay_Duration_Category']

    if stay_duration == 'Short Stay':
        return price * 0.8
    elif stay_duration == 'Medium Stay':
        return price * 1.0
    else:  # Long Stay
        return price * 1.2

# Calculate the numerical 'Impact' column
df['Impact'] = df.apply(calculate_numerical_impact, axis=1)

# Calculate percentiles to define thresholds for impact categories
low_impact_threshold = df['Impact'].quantile(0.33)
high_impact_threshold = df['Impact'].quantile(0.66)

def categorize_numerical_impact(impact_score):
    if impact_score < low_impact_threshold:
        return 'Low Impact'
    elif low_impact_threshold <= impact_score < high_impact_threshold:
        return 'Medium Impact'
    else:
        return 'High Impact'

# Apply the categorization directly to the 'Impact' column, overwriting the numerical 'Impact'
df['Impact'] = df['Impact'].apply(categorize_numerical_impact)
display(df.head())

,SaleId,SaleDate,CheckInDate,Price,ConceptName,SaleCityName,CInDay,SaleCheckInDayDiff,Seasons,Category,Concept_Category,Stay_Duration_Category,Discounted_Price,Impact
0,415122,2022-12-03,2022-12-03,79.304029,Herşey Dahil,Antalya,Saturday,0,Low,Medium Price,All Inclusive,Short Stay,71.373626,Medium Impact
1,415103,2022-12-03,2022-12-03,45.970696,Yarım Pansiyon,Antalya,Saturday,0,Low,Low Price,Half Board,Short Stay,41.373626,Low Impact
2,404034,2022-09-12,2022-09-13,77.838828,Herşey Dahil,Antalya,Tuesday,1,High,Medium Price,All Inclusive,Short Stay,77.838828,Medium Impact
3,415094,2022-12-03,2022-12-10,222.710623,Yarım Pansiyon,İzmir,Saturday,7,Low,High Price,Half Board,Medium Stay,200.439560,High Impact
4,414951,2022-12-01,2022-12-03,140.476190,Yarım Pansiyon,İzmir,Saturday,2,Low,Medium Price,Half Board,Short Stay,126.428571,High Impact
